In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/comment-category-prediction-challenge/Sample.csv
/kaggle/input/comment-category-prediction-challenge/train.csv
/kaggle/input/comment-category-prediction-challenge/test.csv


In [2]:
train = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/train.csv')
test = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/test.csv')
sample_sub = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/Sample.csv')

print(train.shape, test.shape)
train.head()

(198000, 15) (102000, 14)


,created_date,post_id,emoticon_1,emoticon_2,emoticon_3,upvote,downvote,if_1,if_2,race,religion,gender,disability,comment,label
0,2024-01-18 08:43:57.397508+00:00,73,0,0,0,0,1,0,10,NaN,NaN,NaN,False,She might be a bright spot for a party keou on...,2
1,2024-03-24 21:43:11.490017+00:00,39,0,0,0,6,0,0,4,NaN,NaN,NaN,False,"Under Alaska law, a non-tribal member is not b...",0
2,2024-04-24 20:32:17.014931+00:00,31,0,1,1,0,0,0,10,NaN,NaN,NaN,False,in the future please spare me your strawman dr...,2
3,2023-05-28 22:00:14.214527+00:00,39,0,0,0,5,0,0,10,NaN,NaN,NaN,False,"PS: That should have been ""rot"" instead of ""co...",2
4,2023-09-09 23:12:05.689498+00:00,39,0,0,0,0,0,0,10,NaN,NaN,NaN,False,"Today, the confederate flag...tomorrow, the na...",2


In [3]:
print(train.info(), test.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 198000 entries, 0 to 197999
Data columns (total 15 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   created_date  198000 non-null  object
 1   post_id       198000 non-null  int64 
 2   emoticon_1    198000 non-null  int64 
 3   emoticon_2    198000 non-null  int64 
 4   emoticon_3    198000 non-null  int64 
 5   upvote        198000 non-null  int64 
 6   downvote      198000 non-null  int64 
 7   if_1          198000 non-null  int64 
 8   if_2          198000 non-null  int64 
 9   race          52577 non-null   object
 10  religion      52577 non-null   object
 11  gender        52577 non-null   object
 12  disability    198000 non-null  bool  
 13  comment       197999 non-null  object
 14  label         198000 non-null  int64 
dtypes: bool(1), int64(9), object(5)
memory usage: 21.3+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102000 entries, 0 to 101999
Data columns (total 

**Data Preperation**

In [4]:


train = train.copy()
test = test.copy()


train['comment'] = train['comment'].fillna("").astype(str)
test['comment'] = test['comment'].fillna("").astype(str)


train['comment'] = train['comment'].str.lower()
test['comment'] = test['comment'].str.lower()


train['created_date'] = pd.to_datetime(train['created_date'], utc=True)
test['created_date'] = pd.to_datetime(test['created_date'], utc=True)

for df in [train, test]:
    df['month'] = df['created_date'].dt.month.astype('int8')
    df['day'] = df['created_date'].dt.day.astype('int8')
    df['hour'] = df['created_date'].dt.hour.astype('int8')


for df in [train, test]:
    df['vote_diff'] = (df['upvote'] - df['downvote']).astype('int32')
    df['vote_total'] = (df['upvote'] + df['downvote']).astype('int32')
    df['vote_ratio'] = (df['upvote'] / (df['vote_total'] + 1)).astype('float32')


train['disability'] = train['disability'].astype(int)
test['disability'] = test['disability'].astype(int)


for col in ['race', 'religion', 'gender']:
    train[f'has_{col}'] = train[col].notna().astype('int8')
    test[f'has_{col}'] = test[col].notna().astype('int8')


drop_cols = ['created_date', 'race', 'religion', 'gender', 'post_id']

train.drop(columns=drop_cols, inplace=True)
test.drop(columns=drop_cols, inplace=True)

# 8️⃣ Final numeric feature list
num_features = [
    'emoticon_1', 'emoticon_2', 'emoticon_3',
    'upvote', 'downvote',
    'vote_diff', 'vote_total', 'vote_ratio',
    'if_1', 'if_2',
    'disability',
    'has_race', 'has_religion', 'has_gender',
    'month', 'day', 'hour'
]


train[num_features] = train[num_features].fillna(0).astype('float32')
test[num_features] = test[num_features].fillna(0).astype('float32')

print("Preprocessing Complete ✅")
print("Train shape:", train.shape)
print("Test shape:", test.shape)

Preprocessing Complete ✅
Train shape: (198000, 19)
Test shape: (102000, 18)


**FEATURE ENGINEERING**

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    stop_words='english'
)

X_text_train = tfidf.fit_transform(train['comment'])
X_text_test = tfidf.transform(test['comment'])

In [7]:
from scipy.sparse import csr_matrix, hstack
num_features = [
    'emoticon_1', 'emoticon_2', 'emoticon_3',
    'upvote', 'downvote',
    'vote_diff', 'vote_total', 'vote_ratio',
    'if_1', 'if_2',
    'disability',
    'has_race', 'has_religion', 'has_gender',
    'month', 'day', 'hour'
]
X_num_train = csr_matrix(train[num_features].values)
X_num_test = csr_matrix(test[num_features].values)

X_train = hstack([X_text_train, X_num_train])
X_test = hstack([X_text_test, X_num_test])

In [16]:
from sklearn.linear_model import LogisticRegression
y = train['label']
model = LogisticRegression(
    max_iter=500,
    solver='saga',   
    n_jobs=-1
)

model.fit(X_train, y)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


LogisticRegression(max_iter=500, n_jobs=-1, solver='saga')

In [19]:
preds = model.predict(X_test)
preds.shape

1

In [20]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import numpy as np

print("Accuracy:", accuracy_score(y_true, y_pred))
print("F1 Macro:", f1_score(y_true, y_pred, average='macro'))
print("F1 Weighted:", f1_score(y_true, y_pred, average='weighted'))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

NameError: name 'y_true' is not defined

In [ ]:
import lightgbm as lgb
import numpy as np

RANDOM_STATE = 42

y = train['label'].values
num_classes = len(np.unique(y))


train_data = lgb.Dataset(X_train_final, label=y)

params = {
    'objective': 'multiclass',
    'num_class': num_classes,
    'learning_rate': 0.05,
    'num_leaves': 64,
    'min_data_in_leaf': 50,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'verbosity': -1,
    'seed': RANDOM_STATE
}

model = lgb.train(
    params,
    train_data,
    num_boost_round=1500
)

# Predict on already processed test matrix
test_preds = model.predict(X_test_final)
test_preds = np.argmax(test_preds, axis=1)